# Instancia Premier League 2024-2025

Descarga desde Football-Data la base de esta temporada y la usa como conjunto de equipos para resolver un calendario doble Round-Robin nuevo con el modelo compartido `modelo_calendario.py`.

## Qué representa esta instancia

El CSV histórico local de 2024-2025 se usa para obtener y validar los 20 clubes de esa temporada. El modelo genera un calendario nuevo de 38 jornadas y 380 partidos; no reproduce el calendario oficial.

## Flujo del notebook

1. Localiza el proyecto e importa `resolver_instancia` desde `modelo_calendario.py`.
2. Lee los partidos históricos desde `BASES/2024-2025.csv`.
3. Resuelve dos configuraciones dentro de un presupuesto total de 3600 segundos.
4. Compara estado, objetivo, cota, gap, tiempo y nodos; luego muestra el calendario y una muestra de los datos históricos.

## Licencia para ejecutar

Antes de ejecutar la celda Python que resuelve el modelo, activa la licencia de Gurobi desde PowerShell o la terminal de Windows. La lectura del CSV local no requiere conexión a internet.

In [ ]:
# Evaluar la instancia 2024-2025 con GurobiPy en dos configuraciones.
from pathlib import Path
import sys
import time

import pandas as pd
from IPython.display import display

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")
sys.path.insert(0, str(RUTA_PROYECTO))
from modelo_calendario import resolver_instancia

TEMPORADA = "2024-2025"
URL_CSV = str(RUTA_PROYECTO / "BASES" / "2024-2025.csv")
PARAMETROS_MODELO = {
    "clasicos": [],
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "numero_equipos_esperado": 20,
}

configuraciones = {
    "Normal (Gurobi por defecto)": False,
    "Sin cortes ni heurísticas": True,
}

TIEMPO_TOTAL = 3600
MARGEN_SEGUNDOS = 30
inicio = time.monotonic()
resultados = {}

for indice, (nombre, desactivar_cortes_heuristicas) in enumerate(configuraciones.items()):
    pendientes = len(configuraciones) - indice
    disponible = TIEMPO_TOTAL - (time.monotonic() - inicio) - MARGEN_SEGUNDOS

    if disponible < 1:
        print(f"Se omite {nombre}: se agotó el presupuesto de tiempo.")
        break

    tiempo_limite = max(1, int(disponible / pendientes))
    print(f"{nombre}: límite de Gurobi de {tiempo_limite} segundos.")
    resultados[nombre] = resolver_instancia(
        TEMPORADA,
        URL_CSV,
        desactivar_cortes_heuristicas=desactivar_cortes_heuristicas,
        tiempo_limite=tiempo_limite,
        **PARAMETROS_MODELO,
    )

reporte = pd.DataFrame(
    [
        {
            "Configuración": nombre,
            "Estado Gurobi": resultado["estado"],
            "Objetivo (ObjVal)": resultado["objetivo"],
            "Cota (ObjBoundC)": resultado["cota"],
            "Gap (MIPGap)": resultado["gap"],
            "Tiempo (s)": resultado["tiempo"],
            "Nodos (NodeCount)": resultado["nodos"],
        }
        for nombre, resultado in resultados.items()
    ]
)
reporte["Objetivo (ObjVal)"] = reporte["Objetivo (ObjVal)"].fillna("No factible")
reporte["Gap (MIPGap)"] = reporte["Gap (MIPGap)"].fillna("No factible")
estado_map = {
    1: "CARGADO",
    2: "OPTIMO",
    3: "INFACTIBLE",
    4: "INFACTIBLE O NO ACOTADO",
    5: "NO ACOTADO",
    6: "CORTE",
    7: "LIMITE DE ITERACIONES",
    8: "LIMITE DE NODOS",
    9: "LIMITE DE TIEMPO",
    10: "LIMITE DE SOLUCIONES",
    11: "INTERRUMPIDO",
    12: "ERROR NUMERICO",
    13: "SUBOPTIMO",
    14: "EN PROGRESO",
    15: "LIMITE DE OBJETIVO",
    16: "LIMITE DE TRABAJO",
    17: "LIMITE DE MEMORIA",
}
reporte["Estado Gurobi"] = reporte["Estado Gurobi"].map(lambda estado: estado_map.get(estado, estado))
display(reporte)

if "Normal (Gurobi por defecto)" in resultados:
    resultado_normal = resultados["Normal (Gurobi por defecto)"]
    print(f"Partidos leídos desde la base local: {len(resultado_normal['partidos_historicos'])}")
    print(f"Breaks: {resultado_normal['breaks']} | Viajes críticos: {resultado_normal['viajes_criticos']}")
    display(resultado_normal["calendario"])
    display(resultado_normal["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))